# Fund Workstation

## Pre-Analysis Loading

#### Environment Setting

In [1]:
import pandas as pd
import plotly.graph_objects as go
from src.fofproject.fund import Fund, input_monthly_returns, subset_of_funds, compare_funds, get_available_benchmarks, load_benchmarks, assign_benchmarks
from fofproject.batch import plot_cumulative_returns, plot_fund_correlation_heatmap
from src.fofproject.mvo import minimum_variance_analysis
from src.fofproject.load import load_saved_json, init_funds, process_pdfs_in_folder, save_changes_in_fund, merge_funds, rerun_no_perf_files, continue_running

# Names of our portfolio & benchmark indices
our_portfolio = ['TAIREN','HAO','LEXINGTON','LIM','FOREST','WT CHINA','E20','3W GLOBAL','3W CHINA','3W HEALTHCARE','TIMEFOLIO','MONOLITH','PERSEVERANCE','NEO IVY','JH BIOTECH']
our_index = get_available_benchmarks("BENCHMARK.csv")

In [2]:
from dotenv import load_dotenv
from pathlib import Path
load_dotenv(Path("src/fofproject/.env"))


True

#### Load Data via .json

In [3]:
test = load_saved_json(folder_path="input/marex")
funds = init_funds(test, benchmarks=load_benchmarks("BENCHMARK.csv"))


In [ ]:
port = load_saved_json(folder_path="input/individual investigation")
funds = init_funds(port, benchmarks=load_benchmarks("BENCHMARK.csv"))


#### Load Data via .csv

In [4]:
# Use for manually update csv data of our portfolio and indices and merge with json file
portfolios = input_monthly_returns(r"RETURN DATA.csv", benchmark_csv="BENCHMARK.csv")
funds = merge_funds(portfolios, funds)

In [ ]:
# Use for manually input unparsable pdf data and merge with json file 
manual_csv = input_monthly_returns(r"MANUAL INPUTS.csv")
funds = merge_funds( manual_csv, funds)

#### GPT Input Factsheet

##### Parse Everything

In [ ]:
# Do not open the pdf while running, or the renaming process will encounter error
gpt_fund = process_pdfs_in_folder(folder_path="input/greenwoods", save=True)
funds = init_funds(gpt_fund, benchmarks=load_benchmarks("BENCHMARK.csv"))

##### Continue Running Unparsed

In [ ]:
gpt_fund_2 = continue_running(folder_path="input/greenwoods", save=True)
gpt_fund_2 = init_funds(gpt_fund_2, benchmarks=load_benchmarks("BENCHMARK.csv"))

In [ ]:
funds = gpt_fund_2 | funds

##### Re-Run No Performance Funds

In [ ]:
# Target the files with "No Performance Found", and re-run the analysis again
gpt_fund_2 = rerun_no_perf_files(folder_path="input/greenwoods", save=True)

#### Merging Different Input

In [ ]:
# Use the left hand's PERFORMANCE TABLE to update the right hand one
funds = merge_funds(portfolio_csv, funds)

In [ ]:
# Right-hand side has higher priority
funds = funds | test

#### Save the Changes to .json & .csv

In [ ]:
# Careful what fund you are saving to what folder
save_changes_in_fund(funds, folder_path="input/worth_a_look")

In [ ]:
print(funds['ASIAN TECHNOLOGY'].total_max_dd)

## Fund Analysis

#### Dataframe Comparison

In [ ]:
df = compare_funds(funds)
do_not_display = our_index + our_portfolio
df = df[~df["Name"].isin(do_not_display)].reset_index(drop=True)
print(df)


In [ ]:
# Loop through all funds: Sharpe since inception, Vol since inception, Return 2025-01 to 2025-12
rows = []
for name, fund in funds.items():
    try:
        rtn_2025 = fund.cumulative_return("2025-01", "2025-12")
    except Exception:
        rtn_2025 = None
    rows.append({
        "Fund": name,
        "Sharpe (Since Inception)": fund.total_sharpe,
        "Volatility (Since Inception)": fund.total_vol,
        "Return 2025": rtn_2025,
    })

summary_df = pd.DataFrame(rows)
summary_df = summary_df.sort_values("Sharpe (Since Inception)", ascending=False).reset_index(drop=True)
summary_df

In [ ]:
# Column List = ["Name", "One Liner", "Geo Focus", "Strategy", "Asset Class", "Identifier", "IR Contact", "AUM (in Mn USD)", 
#                     "Net Exposure", "Net Return", "Mgmt Fee", "Perf Fee", "Inception Date", "Latest Date", "Month Running",
#                     "# Months", "Cumulative Return", "Annualized Return", "Volatility", "Sharpe Ratio", 
#                     "Sortino Ratio", "Max Drawdown", "Positive Months"]

In [ ]:
# sort by what we think important   
df = df.sort_values(by=["Sharpe Ratio", "Annualized Return"], ascending=[False, False])
# mask and filtered only the wanted funds
mask =  (df["Month Running"] > 6) & (df["Annualized Return"] > 0.12)
df['Worth a Look'] = mask
# Display only the following columns
display_df = df[["Worth a Look","Name", "Sharpe Ratio", "Annualized Return", "AUM (in Mn USD)", "Month Running","One Liner","Max Drawdown" ]]
display_df


#### Save it to CSV

In [ ]:
# Select the column you want to save
output_df = df[["Worth a Look","Name", "Sharpe Ratio", "Annualized Return", "AUM (in Mn USD)", "Month Running","Max Drawdown" ,"Net Exposure","Strategy", "Sector","Contact","Description", "Mgmt Fee", "Perf Fee"]]
output_df.to_csv("output/funds_comparison.csv", index=False)

#### Save it to a list

In [ ]:
exclude_list = our_index + our_portfolio
df = df[~df["Name"].isin(exclude_list)].reset_index(drop=True)
worth_looking = df.loc[df["Worth a Look"], "Name"].tolist()
print(worth_looking)

#### Compare Performance Table

In [ ]:
funds['ONELS'].export_monthly_table(benchmark_fund = funds['MSCI CHINA'], benchmark_name = "MSCI\nChina" ,language = "en", inception_column = True)

#### Plot Cumulative Return

In [14]:

funds_to_be_plot = subset_of_funds(funds,['MSCI WORLD', 'WITH WORLD'] + [f for f in our_portfolio if f != 'HAO MIX' and f != 'RDGFF' and f != 'NEW RDGFF'] + ['HAO'])
# funds_to_be_plot = subset_of_funds(funds,['FOF', 'WITH WORLD', 'MSCI CHINA'])

start_month = "2025-1"
end_month = "2025-12"

plot = plot_cumulative_returns(
    funds=funds_to_be_plot,
    title="",
    start_month=start_month,
    end_month=end_month,
    style=None,
    language="cn",
    blur=False,
    aspect_lock=True,
    custom_ticks=False,
    save=True,
    toggle=False,
    highlight_extremes=3,
    strict_period=True,
    )


strict_period: excluded ['E20'] (insufficient data for 2025-01 to 2025-12)


#### Correlation Heat Map

In [ ]:
# Correlation heatmap
funds_to_be_plot = subset_of_funds(funds, ['QUANTICA', 'RDGFF', 'HAO','US FINANCIAL'])
start_month = "2019-12"
end_month = "2025-7"

fig, corr_df, overlap_df = plot_fund_correlation_heatmap(funds_to_be_plot, method="pearson", min_overlap=12, save=True)
fig.show()

#### Efficient Frontier Analysis

In [ ]:
remove_list = ['RDGFF', 'E20']

for i, k in enumerate(our_portfolio):
    if k in remove_list:
        our_portfolio.pop(i)
test =  ['RDGFF', 'QUANTICA']
print(our_portfolio)


funds_to_be_plot = subset_of_funds(funds, test)
print(len(funds))

In [ ]:
# Choose Mode between "Maximum Sharpe", "Minimum Variance", "Target Return"
fig, weights, stats = minimum_variance_analysis(funds=funds_to_be_plot, mode="Maximum Sharpe", title=None)
print(weights)
print(stats)

#### Summary of a Fund

In [ ]:
# our_index = ['EUREKAHEDGE WORLD', 'MSCI CHINA',	'MSCI WORLD', 'EUREKAHEDGE ASIA', 'TOPIX', 'S&P 500', 'SOX', 'KOSPI', 'TAIEX', 'MSCI EM', 'RUSSELL 2000', 'STOXX 600', 'STOXX 50', 'FTSE UK', 'US HEALTHCARE', 'US FINANCIAL', 'US ENERGY', 'COMMODITY']
fund_name = 'FOF'
funds[fund_name].summary_of_a_fund(benchmark_fund=funds['MSCI CHINA'],language="en",save=True)
funds[fund_name].compare_worst_performance(funds['RDGFF'], title="Performance during our fund's top 10 drawdowns", n_worst=10,  save=True)
funds[fund_name].compare_worst_performance(funds['S&P 500'], title="Entire performance compared with our fund's" ,n_worst=100,  save=True)

#### Correlation Deep Dive

In [36]:
funds['S&P 500'].annualized_return(start_month = "2018-1", end_month = "2025-12") / funds['S&P 500'].max_drawdown(start_month = "2018-1", end_month = "2025-12")

np.float64(0.5999523341677124)

In [ ]:
# fund_name = 'FOREST STD'
funds_to_be_plot = subset_of_funds(funds, [fund_name] + our_index)
fig, corr_df, overlap_df = plot_fund_correlation_heatmap(funds_to_be_plot, method="pearson", min_overlap=12, save=True)
fig.show()

In [ ]:
the_fund_to_investigate = funds['RDGFF']
benchmark_fund = funds['MSCI CHINA']
# The List = ["cagr","vol","sharpe","sortino","mdd","beta","corr","win","best","worst","aum","skew","kurt","turnover"]



fig1 = the_fund_to_investigate.export_key_metrics_table(
            benchmark_fund=benchmark_fund,
            end_month=the_fund_to_investigate.latest_date,
            language="en",
            metrics=["cagr", "vol", "sharpe", "sortino", "mdd", "beta", "corr", "win"],
            horizontal=False,
        )
